In [ ]:
import random
import pandas as pd
import torch
import hashlib
from tqdm import tqdm
from nltk.corpus import wordnet
import nltk

nltk.download("wordnet")

# Carregar o dataset SST
df = pd.read_csv("data/ids-sst-train.csv", delimiter="\t", usecols=["id", "sentence", "sentiment"])
sentences = df["sentence"].tolist()
sentiments = df["sentiment"].tolist()

# Aplicar Data Augmentation
augmented_sentences = []
augmented_sentiments = []

# Configurações
num_synonym_replacements = 1  # Quantas palavras substituir por sinônimos

print(f"\nAplicando Data Augmentation...\n")

for (sentence, sentiment) in tqdm(zip(sentences, sentiments), total=len(sentences), desc="Processando frases", unit="frase"):
    augmented = []

    # Substituição por sinônimos (WordNet)
    words = sentence.split()
    new_words = words.copy()
    
    for _ in range(num_synonym_replacements):
        word_idx = random.randint(0, len(words) - 1)
        synonyms = wordnet.synsets(words[word_idx])

        if synonyms:
            synonym = synonyms[0].lemmas()[0].name().replace("_", " ")
            new_words[word_idx] = synonym

    augmented.append(" ".join(new_words))

  
    for aug_sentence in augmented:
        augmented_sentences.append(aug_sentence)
        augmented_sentiments.append(sentiment)


# Gerar IDs aleatórios para os novos dados (SHA-256 truncado)
augmented_ids = [hashlib.sha256(sent.encode()).hexdigest()[:24] for sent in augmented_sentences]

# Criar um novo DataFrame com os dados aumentados
augmented_df = pd.DataFrame({"id": augmented_ids, "sentence": augmented_sentences, "sentiment": augmented_sentiments})

# Concatenar com o dataset original
final_df = pd.concat([df, augmented_df], ignore_index=True)

# Salvar novo dataset
final_df.to_csv("data/ids-sst-train-augmented.csv", sep="\t", index=False)
print(f"\nData Augmentation concluído! Total de frases: {len(final_df)}")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Aplicando Data Augmentation...



Processando frases: 100%|██████████| 8544/8544 [00:00<00:00, 15174.68frase/s]



Data Augmentation concluído! Total de frases: 17088
